In [0]:
from datetime import datetime
from delta.tables import DeltaTable
from pyspark.sql.functions import lit, current_timestamp, col
from pyspark.sql.types import IntegerType, TimestampType, StringType

In [0]:
df = spark.read.format("csv").option("header", True).load("/Volumes/nyctaxi/00_landing/data_sources/lookup/taxi_zone_lookup.csv")
df = df.select(
    col("LocationID").cast(IntegerType()).alias("location_id"),
    col("Borough").alias("borough"),
    col("Zone").alias("zone"),
    col("service_zone").alias("service_zone"),
    current_timestamp().alias("effective_date"),
    lit(None).cast(TimestampType()).alias("end_date")
)

end_timestamp = datetime.now()

# load scd2 delta table
dt = DeltaTable.forName(spark, "nyctaxi.02_silver.taxi_zone_lookup")



#df.write.mode("overwrite").saveAsTable("nyctaxi.02_silver.taxi_zone_lookup")

In [0]:
# ingesting and updating records to make sure that scd2 logic works

from pyspark.sql.functions import *

df_new = spark.createDataFrame(
    [(999,"New Borogh","new Zone","new Service Zone")],
    schema = "location_id int, borough string , zone string , service_zone string"
). withColumn("effective_date", current_timestamp()).withColumn("end_date", lit(None).cast(TimestampType()))

df = df_new.union(df)

df = df.withColumn("borough", when(col("location_id")==3, "abcd").otherwise(col("borough")))


In [0]:
# 1 - if any existing records have changes, set end_date to them

dt.alias("target").\
    merge(
        source = df.alias("source"),
        condition = "target.location_id = source.location_id AND target.end_date IS NULL AND (target.borough != source.borough OR target.zone != source.zone OR target.service_zone != source.service_zone)"
    ).\
    whenMatchedUpdate(
        set = { "target.end_date": lit(end_timestamp) }
    ).\
    execute()

    


In [0]:
# 2 - add new records for records patched and set as end_date (we need to create new records fro them with end_date null and they become our current records according to scd2)

df_closed_records = dt.toDF().filter(f"end_date = '{end_timestamp}'").select("location_id")


ids_with_end_date = [row.location_id for row in df_closed_records.collect()]

if(len(ids_with_end_date)==0):
    print("No records to be added")
else:
    ids_string = ", ".join(str(int(id)) for id in ids_with_end_date)
    condition = f"s.location_id not in ({ids_string})"
    
    dt.alias("t").\
        merge(
            source = df.alias("s"),
            condition = condition
        ).\
        whenNotMatchedInsert(
            values = { "t.location_id": "s.location_id",
                    "t.borough": "s.borough",
                    "t.zone": "s.zone",
                    "t.service_zone": "s.service_zone",
                    "t.effective_date": current_timestamp(),
                    "t.end_date": lit(None).cast(TimestampType())}
        ).\
        execute()



In [0]:
#3 insert brand new records
dt.alias("t").\
    merge(
        source = df.alias("s"),
        condition = "t.location_id = s.location_id"
    ).\
    whenNotMatchedInsert(
        values = { "t.location_id": "s.location_id",
                    "t.borough": "s.borough",
                    "t.zone": "s.zone",
                    "t.service_zone": "s.service_zone",
                    "t.effective_date": current_timestamp(),
                    "t.end_date": lit(None).cast(TimestampType())}
    ).\
    execute()